In [1]:
import pandas as pd
from pathlib import Path

SCALED_DIR = Path("../Data/Processed")

# Reload all the train/test splits saved at the end of Step 2
X_german_train = pd.read_csv(SCALED_DIR / "german_X_train.csv")
X_german_test = pd.read_csv(SCALED_DIR / "german_X_test.csv")
y_german_train = pd.read_csv(SCALED_DIR / "german_y_train.csv").squeeze()
y_german_test = pd.read_csv(SCALED_DIR / "german_y_test.csv").squeeze()

X_home_train = pd.read_csv(SCALED_DIR / "home_X_train.csv")
X_home_test = pd.read_csv(SCALED_DIR / "home_X_test.csv")
y_home_train = pd.read_csv(SCALED_DIR / "home_y_train.csv").squeeze()
y_home_test = pd.read_csv(SCALED_DIR / "home_y_test.csv").squeeze()

X_lending_train = pd.read_csv(SCALED_DIR / "lending_X_train.csv")
X_lending_test = pd.read_csv(SCALED_DIR / "lending_X_test.csv")
y_lending_train = pd.read_csv(SCALED_DIR / "lending_y_train.csv").squeeze()
y_lending_test = pd.read_csv(SCALED_DIR / "lending_y_test.csv").squeeze()

print("German:", X_german_train.shape, X_german_test.shape)
print("Home:", X_home_train.shape, X_home_test.shape)
print("Lending:", X_lending_train.shape, X_lending_test.shape)

German: (800, 40) (200, 40)
Home: (246005, 100) (61502, 100)
Lending: (8000, 69) (2000, 69)


In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, f1_score, matthews_corrcoef
from imblearn.metrics import geometric_mean_score

def run_baseline_lr(X_train, y_train, X_test, y_test, name):
    # max_iter increased from default (100) since some of these datasets
    # are large/high-dimensional and may not converge otherwise
    model = LogisticRegression(max_iter=1000, random_state=42)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    results = {
        "dataset": name,
        "AUC-ROC": roc_auc_score(y_test, y_proba),
        "F1 (minority)": f1_score(y_test, y_pred),
        "G-mean": geometric_mean_score(y_test, y_pred),
        "MCC": matthews_corrcoef(y_test, y_pred)
    }
    return results, model

results_german, model_german_lr = run_baseline_lr(X_german_train, y_german_train, X_german_test, y_german_test, "German Credit")
results_home, model_home_lr = run_baseline_lr(X_home_train, y_home_train, X_home_test, y_home_test, "Home Credit")
results_lending, model_lending_lr = run_baseline_lr(X_lending_train, y_lending_train, X_lending_test, y_lending_test, "LendingClub")

baseline_results = pd.DataFrame([results_german, results_home, results_lending])
print(baseline_results)

         dataset   AUC-ROC  F1 (minority)    G-mean       MCC
0  German Credit  0.805357       0.543689  0.645497  0.401036
1    Home Credit  0.743673       0.019739  0.100307  0.061684
2    LendingClub  0.852837       0.200000  0.333333  0.330651


In [3]:
# Check how many positive predictions the Home Credit baseline actually made
y_pred_home = model_home_lr.predict(X_home_test)
print(pd.Series(y_pred_home).value_counts())

0    61401
1      101
Name: count, dtype: int64
